In [ ]:
import pymc as pm; import arviz as az; import sys; import numpy as np; import pickle; import pytensor
import nutpie; import pytensor.tensor as pt; import matplotlib.pyplot as plt
sys.path.append(r'C:\Users\awast\OneDrive\Desktop\MKM')
from _CO_Oxidation.config import kb_eV, T, kb_J, h, F, N_A
from _CO_Oxidation import wrapper_base as wp
from _CO_Oxidation.wrapper_base import plot_posteriors, plot_model_fits, plot_coverages, plot_drc, simulate_fake_data

C_KOH_list = np.array([0.25, 0.5, 1]) # in M 
P_CO_list = 0.01*np.array([0.1, 1, 10, 100]) # in atm 
experiments_data = pickle.load(open('import_Pd100_base_replicates.pkl', 'rb'))
wp.process_experimental_data(experiments_data, C_KOH_list, P_CO_list)
E_in = wp.E_in; P_CO_in = wp.P_CO_in; C_KOH_in = wp.C_KOH_in

def fit_and_evaluate(model, draws=1000, tune=2000, chains=4, cores=4, init_mean=None, target_accept=0.9):
    compiled_model = nutpie.compile_pymc_model(model)
    trace = nutpie.sample(compiled_model, draws=draws, tune=tune, chains=chains, cores=cores, init_mean=init_mean, target_accept=target_accept)
    pm.compute_log_likelihood(trace, progressbar=False, model=model)
    loo = az.loo(trace, pointwise=True)
    print(loo); az.plot_khat(loo); plt.title("Pareto k diagnostic"); plt.show()
    trace = wp.add_post_sampling_observables(trace)
    return trace, loo

def observables(log_rate):
    pm.Deterministic('log_rate', log_rate)
    sigma_rel = pm.HalfNormal('sigma_rel', sigma=0.1) # 0.1
    rate = pm.LogNormal('rate', mu=log_rate, sigma=sigma_rel, observed=wp.rate_obs_matrix)
    return rate

Ag_comp = 0.0

# LH ER (Lateral)

In [ ]:
with pm.Model() as LH_ER_lat:
    '''
    1. CO + * <-> CO* (QEA)
    2a. CO* + OH- -> COOH* + (e-) (SSA) (RDS)
    2b. CO* + OH* -> COOH* + * (SSA) (RDS)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    '''
    # Priors
    deltaG1_0 = pm.Normal('deltaG1_0', mu=-0.4, sigma=0.2)
    deltaG4_0 = pm.Normal('deltaG4_0', mu=0.0, sigma=0.2,)
    beta_2 = pm.Uniform('beta_2', lower=0.1, upper=0.9)
    Gact2_ER_0 = pm.Normal('Gact2_ER_0', mu=0.7, sigma=0.2)
    Gact2_LH_0 = pm.Normal('Gact2_LH_0', mu=0.7, sigma=0.2)
    z_CO_CO = pm.Uniform('z_CO_CO', lower=0.0, upper=0.5) # Interaction parameter for CO-CO interactions
    z_OH_OH = pm.Uniform('z_OH_OH', lower=0.0, upper=0.5) # Interaction parameter for OH-OH interactions
    z_cross = pm.Uniform('z_cross', lower=0.0, upper=0.5) # Interaction parameter for CO-OH interactions (symmetric interactions)

    alpha_BEP = 0.65 # BEP parameter 

    curr_theta_CO = pt.zeros_like(E_in) + 0.5
    curr_theta_OH = pt.zeros_like(E_in) + 0.5
    inv_kT = 1.0/(kb_eV*T)
    log_TST = np.log(kb_J*T/h)
    log_C_KOH = np.log(C_KOH_in)
    log_P_CO = np.log(P_CO_in)
    log_K1_base = -deltaG1_0 * inv_kT
    log_k2_ER_base = log_TST - (Gact2_ER_0 - beta_2 * E_in) * inv_kT
    log_k2_LH_base = log_TST - Gact2_LH_0 * inv_kT
    log_K4_base = -(deltaG4_0 - E_in) * inv_kT

    n_iters = 25
    lams = pt.as_tensor_variable(0.85 * (1.0 - np.arange(n_iters) / n_iters) + 0.05)
    
    def fixed_point_step(lam, c_CO, c_OH, log_K1_base, log_K4_base, log_P_CO, log_C_KOH, inv_kT, z_CO_CO, z_OH_OH, z_cross):
        lat_CO = z_CO_CO * c_CO + z_cross * c_OH
        lat_OH = z_OH_OH * c_OH + z_cross * c_CO
        log_ratio_OH = log_K4_base + log_C_KOH - lat_OH * inv_kT
        log_ratio_CO = log_K1_base + log_P_CO - lat_CO * inv_kT
        log_sum_terms = pt.logaddexp(0.0, pt.logaddexp(log_ratio_CO, log_ratio_OH))
        log_theta_empty_calc = -log_sum_terms
        target_theta_CO = pt.exp(log_ratio_CO + log_theta_empty_calc)
        target_theta_OH = pt.exp(log_ratio_OH + log_theta_empty_calc)
        next_theta_CO = (1.0 - lam) * c_CO + lam * target_theta_CO
        next_theta_OH = (1.0 - lam) * c_OH + lam * target_theta_OH
        return next_theta_CO, next_theta_OH

    outputs = pytensor.scan(fn = fixed_point_step, n_steps = n_iters, sequences = [lams], outputs_info = [curr_theta_CO, curr_theta_OH], return_updates=False, 
        non_sequences = [log_K1_base, log_K4_base, log_P_CO, log_C_KOH, inv_kT, z_CO_CO, z_OH_OH, z_cross])
    curr_theta_CO_root = pytensor.gradient.disconnected_grad(outputs[0][-1])
    curr_theta_OH_root = pytensor.gradient.disconnected_grad(outputs[1][-1])
    prev_theta_CO_root = pytensor.gradient.disconnected_grad(outputs[0][-2])
    prev_theta_OH_root = pytensor.gradient.disconnected_grad(outputs[1][-2])
    pm.Deterministic('CO_converge_error', pt.max(pt.abs(curr_theta_CO_root - prev_theta_CO_root)))
    pm.Deterministic('OH_converge_error', pt.max(pt.abs(curr_theta_OH_root - prev_theta_OH_root)))
    curr_theta_CO_final, curr_theta_OH_final = fixed_point_step(pt.as_tensor_variable(1.0), curr_theta_CO_root, curr_theta_OH_root, 
        log_K1_base, log_K4_base, log_P_CO, log_C_KOH, inv_kT, z_CO_CO, z_OH_OH, z_cross)
    
    theta_CO = pm.Deterministic('theta_CO', curr_theta_CO_final)
    theta_OH = pm.Deterministic('theta_OH', curr_theta_OH_final)
    theta_empty = pm.Deterministic('theta_empty', 1.0 - theta_CO - theta_OH)
    lat_CO = z_CO_CO * theta_CO + z_cross * theta_OH
    lat_OH = z_OH_OH * theta_OH + z_cross * theta_CO
    log_k2_ER = log_k2_ER_base + alpha_BEP * lat_CO * inv_kT
    log_k2_LH = log_k2_LH_base + alpha_BEP * (lat_CO + lat_OH) * inv_kT

    # Rate expression
    log_rate_ER = log_k2_ER + pt.log(theta_CO) + log_C_KOH
    log_rate_LH = log_k2_LH + pt.log(theta_CO) + pt.log(theta_OH) + np.log(1-Ag_comp)
    log_rate = pt.logaddexp(log_rate_ER, log_rate_LH)
    rate = observables(log_rate)
    
trace_LH_ER_lat, loo_LH_ER_lat = fit_and_evaluate(LH_ER_lat, tune=500, draws=500)

ppc_LH_ER_lat = plot_posteriors(trace_LH_ER_lat, LH_ER_lat)
plot_model_fits(trace_LH_ER_lat, ppc_LH_ER_lat, loo_LH_ER_lat); plot_coverages(trace_LH_ER_lat)
plot_drc(LH_ER_lat, trace_LH_ER_lat, perturb_vars=['Gact2_ER_0', 'Gact2_LH_0'], perturb_labels=['ER', 'LH'])

# CO LH ER OH (Lateral) (?)
Even better ELPD, but all the repulsion goes into OH-OH. CO is not repulsed by other CO's. Increase in ELPD has to be coming from mathematical flexibility. <br>
CO coverages are super low. CO adsorption energies are not better. Weird DRCs. OH ads activation energy is too high. <br> 
Current priors are narrow for faster draws while running, but comments reflect actual priors that were used to get to the narrow priors. <br>
Takes forever to run, not going to run any more lateral interaction models. <br>

In [ ]:
with pm.Model() as CO_LH_ER_OH_lat:
    '''
    1. CO + * <-> CO* (SSA)
    2a. CO* + OH- -> COOH* + (e-) (SSA) (RDS)
    2b. CO* + OH* -> COOH* + * (SSA) (RDS)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (SSA)
    '''
    # Priors
    deltaG1_0 = pm.TruncatedNormal('deltaG1_0', mu=-0.1, sigma=0.2, lower=-0.2, upper=0) # Forcing CO ads energy to be smaller than 0 eV 
    deltaG4_0 = pm.TruncatedNormal('deltaG4_0', mu=-0.3, sigma=0.2, lower=-0.5, upper=0) # Forcing OH ads energy to be smaller than 0.3 eV 
    beta_2 = pm.Uniform('beta_2', lower=0.1, upper=0.5) # Preventing values close to 0 and 1
    beta_4 = pm.Uniform('beta_4', lower=0.3, upper=0.5) # Preventing values close to 0 and 1
    Gact1_0 = pm.TruncatedNormal('Gact1_0', mu=0.6, sigma=0.2, lower=0.55, upper=0.6) # CO ads activation energy limited between 0.4 eV and 0.8 eV
    Gact2_ER_0 = pm.TruncatedNormal('Gact2_ER_0', mu=0.8, sigma=0.2, lower=0.75, upper=0.9) # ER activation energy limited between 0.5 eV and 0.8 eV
    Gact2_LH_0 = pm.TruncatedNormal('Gact2_LH_0', mu=0.8, sigma=0.2, lower=0.75, upper=0.9) # LH activation energy limited between 0.6 eV and 0.9 eV
    Gact4_0 = pm.TruncatedNormal('Gact4_0', mu=0.8, sigma=0.2, lower=0.75, upper=0.8) # OH ads activation energy limited between 0.5 eV and 0.8 eV
    z_CO_CO = pm.Uniform('z_CO_CO', lower=0.0, upper=0.1) # Interaction parameter for CO-CO interactions
    z_OH_OH = pm.Uniform('z_OH_OH', lower=0.3, upper=0.7) # Interaction parameter for OH-OH interactions
    z_cross = pm.Uniform('z_cross', lower=0.1, upper=0.3) # Interaction parameter for CO-OH interactions (symmetric interactions)

    alpha_BEP = 0.65 # BEP parameter 

    curr_theta_CO = pt.zeros_like(E_in) + 0.5
    curr_theta_OH = pt.zeros_like(E_in) + 0.5
    inv_kT = 1.0/(kb_eV*T)
    log_TST = np.log(kb_J*T/h)
    log_C_KOH = np.log(C_KOH_in)
    log_P_CO = np.log(P_CO_in)
    log_k1_base = log_TST - Gact1_0 * inv_kT
    log_k_minus1_base = log_k1_base + deltaG1_0 * inv_kT
    log_k2_ER_base = log_TST - (Gact2_ER_0 - beta_2 * E_in) * inv_kT
    log_k2_LH_base = log_TST - Gact2_LH_0 * inv_kT
    log_K4_base = -(deltaG4_0 - E_in) * inv_kT 
    log_k4_base = log_TST - (Gact4_0 - beta_4 * E_in) * inv_kT
    log_k_minus4_base = log_k4_base - log_K4_base
    log_k1_P_CO = log_k1_base + log_P_CO
    log_k4_C_KOH = log_k4_base + log_C_KOH

    n_iters = 25
    lams = pt.as_tensor_variable(0.85 * (1.0 - np.arange(n_iters) / n_iters) + 0.05)
    
    def fixed_point_step(lam, c_CO, c_OH, log_k1_P_CO, log_k4_C_KOH, log_k_minus4_base, log_k_minus1_base, log_k2_ER_base, log_C_KOH, log_k2_LH_base, inv_kT, z_CO_CO, z_OH_OH, z_cross):
        lat_CO = z_CO_CO * c_CO + z_cross * c_OH
        lat_OH = z_OH_OH * c_OH + z_cross * c_CO
        log_k_minus4_lat = log_k_minus4_base + lat_OH * inv_kT 
        log_k2_LH_OH_consumption = log_k2_LH_base + pt.log(c_CO) + alpha_BEP * (lat_CO + lat_OH) * inv_kT
        log_denom_OH = pt.logaddexp(log_k_minus4_lat, log_k2_LH_OH_consumption)
        log_ratio_OH = log_k4_C_KOH - log_denom_OH
        log_k_minus1_lat = log_k_minus1_base + lat_CO * inv_kT
        log_k2_ER_CO_consumption = log_k2_ER_base + log_C_KOH + alpha_BEP * lat_CO * inv_kT
        log_denom_static = pt.logaddexp(log_k_minus1_lat, log_k2_ER_CO_consumption)
        log_cons_LH_rel = log_k2_LH_base + pt.log(c_OH) + alpha_BEP * (lat_CO + lat_OH) * inv_kT
        log_denom_CO_rel = pt.logaddexp(log_denom_static, log_cons_LH_rel)
        log_ratio_CO = log_k1_P_CO - log_denom_CO_rel
        log_sum_terms = pt.logaddexp(0.0, pt.logaddexp(log_ratio_CO, log_ratio_OH))
        log_theta_empty_calc = -log_sum_terms
        target_theta_CO = pt.exp(log_ratio_CO + log_theta_empty_calc)
        target_theta_OH = pt.exp(log_ratio_OH + log_theta_empty_calc)
        next_theta_CO = (1.0 - lam) * c_CO + lam * target_theta_CO
        next_theta_OH = (1.0 - lam) * c_OH + lam * target_theta_OH
        return next_theta_CO, next_theta_OH

    outputs = pytensor.scan(fn = fixed_point_step, n_steps = n_iters, sequences = [lams], outputs_info = [curr_theta_CO, curr_theta_OH], return_updates=False,
        non_sequences = [log_k1_P_CO, log_k4_C_KOH, log_k_minus4_base, log_k_minus1_base, log_k2_ER_base, log_C_KOH, log_k2_LH_base, inv_kT, z_CO_CO, z_OH_OH, z_cross])
    curr_theta_CO_root = pytensor.gradient.disconnected_grad(outputs[0][-1])
    curr_theta_OH_root = pytensor.gradient.disconnected_grad(outputs[1][-1])
    prev_theta_CO_root = pytensor.gradient.disconnected_grad(outputs[0][-2])
    prev_theta_OH_root = pytensor.gradient.disconnected_grad(outputs[1][-2])
    pm.Deterministic('CO_converge_error', pt.max(pt.abs(curr_theta_CO_root - prev_theta_CO_root)))
    pm.Deterministic('OH_converge_error', pt.max(pt.abs(curr_theta_OH_root - prev_theta_OH_root)))
    curr_theta_CO_final, curr_theta_OH_final = fixed_point_step(pt.as_tensor_variable(1.0), 
        curr_theta_CO_root, curr_theta_OH_root, log_k1_P_CO, log_k4_C_KOH, log_k_minus4_base, log_k_minus1_base, log_k2_ER_base, log_C_KOH, log_k2_LH_base, inv_kT, z_CO_CO, z_OH_OH, z_cross)
    
    theta_CO = pm.Deterministic('theta_CO', curr_theta_CO_final)
    theta_OH = pm.Deterministic('theta_OH', curr_theta_OH_final)
    theta_empty = pm.Deterministic('theta_empty', 1.0 - theta_CO - theta_OH)
    lat_CO = z_CO_CO * theta_CO + z_cross * theta_OH
    lat_OH = z_OH_OH * theta_OH + z_cross * theta_CO
    log_k2_ER = log_k2_ER_base + alpha_BEP * lat_CO * inv_kT
    log_k2_LH = log_k2_LH_base + alpha_BEP * (lat_CO + lat_OH) * inv_kT

    # Rate expression
    log_rate_ER = log_k2_ER + pt.log(theta_CO) + log_C_KOH
    log_rate_LH = log_k2_LH + pt.log(theta_CO) + pt.log(theta_OH) + np.log(1-Ag_comp)
    log_rate = pt.logaddexp(log_rate_ER, log_rate_LH)
    rate = observables(log_rate)
    
trace_CO_LH_ER_OH_lat, loo_CO_LH_ER_OH_lat = fit_and_evaluate(CO_LH_ER_OH_lat, tune=500, draws=500)

ppc_CO_LH_ER_OH_lat = plot_posteriors(trace_CO_LH_ER_OH_lat, CO_LH_ER_OH_lat)
plot_model_fits(trace_CO_LH_ER_OH_lat, ppc_CO_LH_ER_OH_lat, loo_CO_LH_ER_OH_lat); plot_coverages(trace_CO_LH_ER_OH_lat)
plot_drc(CO_LH_ER_OH_lat, trace_CO_LH_ER_OH_lat, perturb_vars=['Gact2_ER_0', 'Gact2_LH_0', 'Gact1_0', 'Gact4_0'], perturb_labels=['ER', 'LH', 'CO_ads', 'OH ads'])